[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb7_diag.ipynb)

# nb7: joint-run diagnostics (CPU, no training)
Reads the saved `samples_ckpt_*.npz` of `joint_s0` (the run need not be finished; it only reads, never writes there) and asks: (1) physics on all clips vs valid-only clips; (2) **within-checkpoint shuffle baseline** (generated video i with generated sensors j, 20 permutations): did the joint model learn ANY coupling?; (3) which term of the energy balance fails (dE vs work vs dissipation), gyro/torque scale vs real, video-vs-gyro lag; (4) a figure of theta_video vs theta0 + cumint(gyro). Works on a CPU runtime (Runtime -> Change runtime type -> CPU). Writes new files `eval_diag/diag.json`, `diag.md`, `fig_diag_theta.{pdf,png}` (use a new `--tag` to rerun).

## 1. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

## 2. Code

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

## 3. Install

In [ ]:
!pip install -q scipy matplotlib

## 4. Check inputs

In [ ]:
import glob, os
assert os.path.exists(f"{ROOT}/cache/pendulum_T64_dt0.04_heldout.npz"), "missing held-out cache: run nb0_benchmark first"
ck = sorted(glob.glob(f"{ROOT}/joint_s0/samples_ckpt_*.npz"))
print(len(ck), "checkpoint sample files:", [os.path.basename(p) for p in ck])
assert ck, "no joint_s0 samples_ckpt_*.npz found"

## 5. Run diagnostics

In [ ]:
!cd /content/repo && PYTHONUNBUFFERED=1 python -u -m cvpr27.diag --run joint_s0 --tag diag

## 6. Report and figure

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{ROOT}/eval_diag/diag.md").read()))
display(Image(f"{ROOT}/eval_diag/fig_diag_theta.png"))